In [13]:
import torch

from src.utils.data import generate
from src.data.coalitions import CoalitionNN


_Dm, A, C, v, Sh, n = generate(3, 9)
Dm = torch.from_numpy(_Dm[:, 1:])
C = torch.from_numpy(C)

xDm = Dm.unsqueeze(1).repeat(1, 8, 1, 1)
xDm = xDm.reshape(-1, 12 * 4)
xC = C.reshape(-1, 3)
print(xDm.shape)
print(xC)

model = CoalitionNN()
model.load_state_dict(torch.load("src/data/coalitions.pth"))
model.eval()

with torch.no_grad():
    outputs = model(xDm.float(), xC.float())

print('predicted: ', outputs.t())
print('actual: ', v)

#TODO:
#- capture gradients -> helps identify problems
#- change init (need seeding)
#- changing the seed can improve the results (need at least 3 diff seeds) -> find literature to put in thesis
#- make plots of the output to compare predicted to actual

torch.Size([8, 48])
tensor([[0, 0, 0],
        [1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 1, 0],
        [1, 0, 1],
        [0, 1, 1],
        [1, 1, 1]], dtype=torch.int32)
predicted:  tensor([[0.0240, 0.0223, 0.0242, 0.0213, 0.4734, 0.4747, 0.4778, 1.5398]])
actual:  [0.     0.     0.     0.     0.4242 0.6387 0.2219 1.4941]


In [35]:
from bargain.networks import GainNN
from bargain import generate_observation
import torch
import numpy as np

instances, _, coalitions, values, _, _ = generate_observation(n_depots = 3,
                                                              n_customers = 9,
                                                              radius = 0.3)
model = GainNN()
instances = torch.from_numpy(instances).unsqueeze(0)
coalitions = torch.from_numpy(np.array(coalitions)).unsqueeze(0)
values = torch.from_numpy(np.array(values)).unsqueeze(0)
print(instances.shape, coalitions.shape, values.shape)
instances, coalitions, values = model.transform(instances,
                                                coalitions,
                                                values,
                                                device = torch.device("cpu"))

print(instances.shape, coalitions.shape, values.shape)

with torch.no_grad():
    outputs = model(instances.float(), coalition.float())

torch.Size([1, 12, 5]) torch.Size([1, 8, 3]) torch.Size([1, 8])
torch.Size([8, 48]) torch.Size([8, 3]) torch.Size([8, 1])


RuntimeError: Tensors must have same number of dimensions: got 2 and 1